# Model cookbook

A one-page, indexed gallery of pulsar-timing model recipes — from a bare white-noise model up to a full PTA model with a Hellings–Downs background, a decentered parameterisation, and a continuous-wave signal.

Each recipe is a small function that assembles a Discovery likelihood from the public API. **These are the exact models the parity test-suite exercises** (`tests/metamatrix/`): they are defined once in `discovery.recipes` and imported both there and by the tests, so every recipe on this page is covered by a test that asserts the `matrix` and `metamath` kernel backends agree.

Every recipe works unchanged under either backend — flip between them with `ds.config(kernels='matrix')` and `ds.config(kernels='metamath')` (see the last section).

## Index

**Single-pulsar** (`PulsarLikelihood`): `measurement_simple`, `measurement_white`, `ecorr_gp`, `ecorr_sm`, `meas_timing`, `full_rn`, `full_rn_concat_false`, `multi_vgp`, `variable_timing`, `fftcov_2d`, `delay`, `fourier_variance_fixed`

**Multi-pulsar — `GlobalLikelihood`**: `no_global`, `global_hd`, `global_monopole`, `global_compound`

**Multi-pulsar — `ArrayLikelihood`**: `no_common`, `intrinsic_rn`, `intrinsic_plus_crn`, `intrinsic_rn_plus_global_hd`, `decenter_intrinsic_rn`, `decenter_intrinsic_rn_global_hd`, `means_on_commongp`, `extsignal_cw`

In [ ]:
import inspect
from pathlib import Path

import numpy as np
import jax
jax.config.update('jax_enable_x64', True)

import discovery as ds
import discovery.recipes as R   # the model zoo (also used by the parity tests)
from IPython.display import Markdown, display

# bundled example pulsars
datapath = Path(ds.__path__[0]) / '../../data'
files = sorted(datapath.glob('v1p1*.feather'))
psr = ds.Pulsar.read_feather(files[0])          # one pulsar, for single-pulsar recipes
psrs = [ds.Pulsar.read_feather(f) for f in files[:3]]   # a few, for multi-pulsar recipes
print(f'single: {psr.name};  array: {[p.name for p in psrs]}')

In [ ]:
def show(recipe, arg):
    """Render one recipe: caption (its docstring), source, sampled params, and a logL value."""
    caption = ' '.join((recipe.__doc__ or '').split())
    display(Markdown(f"#### `{recipe.__name__}`\n\n{caption}"))
    display(Markdown(f"```python\n{inspect.getsource(recipe)}\n```"))

    model = recipe(arg)
    params = list(model.logL.params)
    display(Markdown(f"- **sampled (hyper)parameters** ({len(params)}): "
                     + (', '.join(f'`{p}`' for p in params) if params else '_none_')))
    try:
        np.random.seed(0)
        p0 = ds.sample_uniform(params) if params else {}
        display(Markdown(f"- log-likelihood at a random draw: `{float(model.logL(p0)):.3f}`"))
    except Exception as e:
        display(Markdown(f"- _logL eval skipped ({type(e).__name__}: needs manual params)_"))

## Single-pulsar models — `PulsarLikelihood`

A `PulsarLikelihood` is a list whose first entry is the residuals and the rest are noise / GP / delay components. Ordered roughly from simplest to most featureful.

In [ ]:
for recipe in R.SINGLE_PULSAR:
    show(recipe, psr)

## Multi-pulsar — `GlobalLikelihood`

`GlobalLikelihood` wraps a list of per-pulsar `PulsarLikelihood`s and an optional `globalgp` for a signal correlated *across* pulsars (e.g. a Hellings–Downs background).

In [ ]:
for recipe in R.GLOBAL:
    show(recipe, psrs)

## Multi-pulsar — `ArrayLikelihood`

`ArrayLikelihood` is the vectorised multi-pulsar likelihood. It adds `commongp` (a shared-basis process across pulsars), `globalgp` (cross-pulsar correlated), `decenter` (whitened-coefficient sampling), per-GP `means`, and `extsignals` (deterministic signals on their own basis, e.g. a continuous wave).

In [ ]:
for recipe in R.ARRAY:
    show(recipe, psrs)

## Switching kernel backends

The same recipe runs on either kernel implementation; `ds.config(kernels=...)` selects it before you build the model. The two agree to numerical precision (this equivalence is what the parity suite certifies).

In [ ]:
np.random.seed(0)
p0 = ds.sample_uniform(R.full_rn(psr).logL.params)

ds.config(kernels='matrix')
v_matrix = float(R.full_rn(psr).logL(p0))

ds.config(kernels='metamath')
v_metamath = float(R.full_rn(psr).logL(p0))

ds.config(kernels='matrix')   # restore default
print(f'full_rn logL  matrix={v_matrix:.6f}  metamath={v_metamath:.6f}  diff={abs(v_matrix-v_metamath):.2e}')